#Unzip folder

In [ ]:
!unzip 'images.zip' -d images/

#Import Libraries

In [3]:
import cv2 as cv
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np


#Get Data

In [4]:
# Crear las listas archivos de audio y su etiqueta correspondiente

from tensorflow.keras.utils import to_categorical
import glob
import os

def getExamples(datafolder):
  X_Image = []
  Y_Classification = []

  # Clasificaciones de clases
  image_classes = [os.path.basename(x) for x in glob.glob(datafolder + '*')]

  for i, image_class in enumerate(image_classes):
    for file in glob.glob(os.path.join(datafolder, image_class) + '/*.jpg'):
      X_Image.append(file)
      Y_Classification.append(np.array(to_categorical(i, num_classes=len(image_classes)),dtype=np.float32))
  return np.asarray(X_Image), np.asarray(Y_Classification)

In [ ]:
datafolder= '/content/images/'
X_Image, Y_Class = getExamples(datafolder)
print(len(X_Image), len(Y_Class))

In [ ]:
print(X_Image[300])
print(Y_Class[300])

In [ ]:
from sklearn.model_selection import train_test_split

X_Image, X_Image_test, Y_Class, Y_Class_test = train_test_split(X_Image, Y_Class, test_size=0.25)
print(len(X_Image))
print(len(Y_Class))

#Convert file paths to tensors

In [5]:
def loadExample(example):
  # Cargar la imagen
  img = tf.io.read_file(example)
  img = tf.image.decode_jpeg(img, channels=3)
  img = tf.image.resize(img, [48,64], preserve_aspect_ratio=True)
  img = tf.image.rgb_to_grayscale(img);
  img = tf.image.convert_image_dtype(img, dtype=tf.float32)
  #img = img/255.0
  img = tf.reshape(img, [img.shape[0], img.shape[1], 1]);

  return img

In [ ]:
s = loadExample(X_Image[0])
s.shape

In [6]:
# Implementar el generador de Datos

class MySequence(tf.keras.utils.Sequence):

  def __init__(self, x_image, y_class, batch_size):
    self.x_image = x_image
    self.y_class = y_class
    self.batch_size = batch_size

  def __len__(self):
    return len(self.x_image)//self.batch_size

  def __getitem__(self, idx):

    batch_y = self.y_class[idx * self.batch_size : (idx+1)*self.batch_size]
    batch_x = np.zeros((self.batch_size, s.shape[0], s.shape[1], s.shape[2]), dtype=np.float32)
    for i in range(0, self.batch_size):
      batch_x[i] = loadExample(self.x_image[idx * self.batch_size + i])
    return batch_x, batch_y

In [ ]:
# Verificar la forma de los datos de entrada y la salida esperada

mS=MySequence(X_Image, Y_Class, 32)
my_data=iter(mS)
bx, by = next(my_data)
print(bx.shape, by.shape)

#Import Keras Libraries

In [8]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Conv2D, BatchNormalization
from tensorflow.keras.layers import MaxPooling2D, Dropout, Flatten
from tensorflow import keras

#Create Model

In [9]:

input_tensor = Input(shape=(48, 64,1))
x = Conv2D(32, 3, activation='relu')(input_tensor)
x = MaxPooling2D()(x)
x = Conv2D(32, 3, activation='relu')(x)
x = MaxPooling2D()(x)
x = Flatten()(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.5)(x)
output_tensor = Dense(2, activation='softmax')(x)

model = Model(inputs=input_tensor, outputs=output_tensor)

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['acc'])



2026-04-05 09:22:12.833699: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /home/lozano/.local/lib/python3.10/site-packages/cv2/../../lib64:
2026-04-05 09:22:12.833734: W tensorflow/stream_executor/cuda/cuda_driver.cc:269] failed call to cuInit: UNKNOWN ERROR (303)
2026-04-05 09:22:12.833756: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (lozano-Precision-7520): /proc/driver/nvidia/version does not exist
2026-04-05 09:22:12.834018: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


#Watch Graph

In [10]:
model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 48, 64, 1)]       0         
                                                                 
 conv2d (Conv2D)             (None, 46, 62, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2D  (None, 23, 31, 32)       0         
 )                                                               
                                                                 
 conv2d_1 (Conv2D)           (None, 21, 29, 32)        9248      
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 10, 14, 32)       0         
 2D)                                                             
                                                                 
 flatten (Flatten)           (None, 4480)              0     

In [11]:
keras.utils.plot_model(model, to_file="model.png", show_shapes=True)

You must install pydot (`pip install pydot`) and install graphviz (see instructions at https://graphviz.gitlab.io/download/) for plot_model/model_to_dot to work.


#Training

In [ ]:
batch_size = 16
epochs = 1
h = model.fit(MySequence(X_Image, Y_Class, batch_size),
              shuffle=True,
              batch_size=batch_size,
              epochs=epochs,
              validation_data=MySequence(X_Image_test, Y_Class_test, batch_size))

In [ ]:
def plotExample(example):
  # Cargar la imagen
  img = tf.io.read_file(example)
  img = tf.image.decode_jpeg(img, channels=3)
  plt.imshow(img)

#Make Prediction

In [ ]:
i=5
print(Y_Class_test[i])
print(X_Image_test[i])

plotExample(X_Image_test[i])
tensor = loadExample(X_Image_test[i])
tensor = tf.reshape(tensor, shape=[1,48,64,1])

model.predict(tensor)

In [ ]:
model.save('image_cnn.h5')

In [ ]:
from keras.models import load_model
smodel=load_model('image_cnn.h5')

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(smodel)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open('image_cnn.tflite', 'wb') as f:
  f.write(tflite_model)